In [137]:
import numpy as np
import numba

def new_shape(arr):
    return tuple([i for i in arr])
    
@numba.njit
def reshaping(arr):
    #new_shape = new_shape(arr)
    return np.arange(24).reshape(arr)

x = np.array([2, 3, 4])


reshaping(tuple(x))

array([[[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]],

       [[12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23]]])

In [48]:
@numba.njit
def test_divmod(x, y):
    return x // y

test_divmod(10, np.array([1, 2, 3]))

array([10,  5,  3])

In [81]:
@numba.njit
def test_unravel_index(x, y):
    return np.unravel_index(x, y)

test_unravel_index(1, np.array([1, 2, 3]))

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
Use of unsupported NumPy function 'numpy.unravel_index' or unsupported use of the function.

File "../../../../../tmp/ipykernel_92031/914115160.py", line 3:
<source missing, REPL/exec in use?>

During: typing of get attribute at /tmp/ipykernel_92031/914115160.py (3)

File "../../../../../tmp/ipykernel_92031/914115160.py", line 3:
<source missing, REPL/exec in use?>


In [54]:
num_kpoints = np.array([2, 2, 1])
nkpts = np.prod(num_kpoints)
ekd = np.zeros((nkpts,nkpts), dtype=np.int16)
for ki in range(nkpts):
    for kip in range(nkpts):
        ind_mat = np.arange(nkpts).reshape(num_kpoints)
        mip = np.array(np.where(ind_mat == kip))
        mi = np.array(np.where(ind_mat == ki))
        md = tuple(mi-mip)
        ekd[ki, kip] = ind_mat[md]
print(ekd)

[[0 1 2 3]
 [1 0 3 2]
 [2 3 0 1]
 [3 2 1 0]]


In [ ]:
@numba.njit
def unravel_index(indices, shape):
    
    

In [100]:
#@numba.njit
def unravel_index(indices, shape, order='C'):
    indices = np.asarray(indices)
    shape = np.asarray(shape)
    if indices.ndim > 1:
        raise ValueError('indices has to be one dimensional')
    if indices.size == 0:
        return np.empty((0, shape.size), dtype=np.intp)
    if indices.dtype != np.intp:
        indices = indices.astype(np.intp)
    if shape.size == 0:
        raise ValueError('an empty shape is not allowed')
    if order == 'C':
        # Use slower np.concatenate because np.r_ is not available in Numba
        strides = np.concatenate(([1], np.cumprod(shape[:0:-1])))[::-1]
        # strides = np.r_[shape[:0:-1].cumprod()[::-1], 1]
    elif order == 'F':
        # Use slower np.concatenate because np.r_ is not available in Numba
        strides = np.concatenate(([1], np.cumprod(shape[1:])))
        # strides = np.r_[1, shape[:-1].cumprod()]
    else:
        raise ValueError("order not understood")
    return np.broadcast_to(indices, (strides.size, indices.size)).T // strides % shape
    

In [101]:
unravel_index(1, np.array([1, 2, 3]))

array([[0, 0, 1]])

In [88]:
def ravel_multi_index(multi_index, dims, mode='raise', order='C'):
    multi_index = np.asarray(multi_index)
    dims = np.asarray(dims)
    if multi_index.ndim > 2:
        raise ValueError('Index has more than two dimensions.')
    if multi_index.size == 0:
        return np.array([], dtype=np.intp)
    if multi_index.ndim == 1:
        multi_index = multi_index[np.newaxis, :]
    if multi_index.dtype != np.intp:
        multi_index = multi_index.astype(np.intp)
    if mode not in ('raise', 'wrap', 'clip'):
        raise ValueError('mode not understood')
    if order not in ('C', 'F'):
        raise ValueError("order not understood")
    if np.any((multi_index < 0) | (multi_index >= dims[np.newaxis, :])):
        if mode == 'raise':
            raise ValueError(
                "invalid entry in coordinates array")
        elif mode == 'clip':
            multi_index = np.clip(multi_index, 0, np.array(dims) - 1)
        elif mode == 'wrap':
            multi_index %= dims[np.newaxis, :]
    if order == 'C':
        strides = np.r_[dims[:0:-1].cumprod()[::-1], 1]
    elif order == 'F':
        strides = np.r_[1, dims[:-1].cumprod()]
    return np.dot(multi_index, strides)

In [90]:
@numba.njit
def numba_unravel_index(indices, shape, order='C'):
    return unravel_index(indices, shape, order)

@numba.njit
def numba_ravel_multi_index(multi_index, dims, mode='raise', order='C'):
    return ravel_multi_index(multi_index, dims, mode, order)

In [97]:
numba_unravel_index(1, np.array([1, 2, 3]))

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'unravel_index': Cannot determine Numba type of <class 'function'>

File "../../../../../tmp/ipykernel_92031/966965266.py", line 3:
<source missing, REPL/exec in use?>


In [91]:
# other way of calculating ekd
# xy (row major ordering)
for ki in range(nkpts):
    for kip in range(nkpts):
        # Row major ordering
        cord1 = numba_unravel_index(ki, num_kpoints, order='C')
        cord2 = numba_unravel_index(kip, num_kpoints, order='C')
        ekd[ki, kip] = np.ravel_multi_index(cord1-cord2, num_kpoints, mode='wrap', order='C')
#print(ekd)

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'unravel_index': Cannot determine Numba type of <class 'function'>

File "../../../../../tmp/ipykernel_92031/966965266.py", line 3:
<source missing, REPL/exec in use?>


In [71]:
rest = -1 % num_kpoints
rest[:-1] @ num_kpoints[1:]


3

In [68]:
rest

array([1, 1, 0])

In [70]:
num_kpoints

array([2, 2, 1])

In [45]:
divmod(-1,np.array([1,2,1]))

(array([-1, -1, -1]), array([0, 1, 0]))

In [31]:
divmod(-8, 3)

(-3, 1)

In [49]:
import numpy as np
x = np.arange(16).reshape((2,8))

i1 = 0
i1 = i1 - 8

x[:, i1+2:i1:-1]

array([[ 2,  1],
       [10,  9]])

In [47]:
5 // np.array([1, 2, 3])

array([5, 2, 1])

In [143]:
def fibunacci(n):
    if n <= 1:
        return n
    else:
        return(fibunacci(n-1) + fibunacci(n-2))

def unique_symmetric_index(i: int, j: int) -> int:
    min_ind = min(i, j)
    max_ind = max(i, j)
    diff = max_ind - min_ind
    return fibunacci(min_ind) + diff * (min_ind + 1 + max_ind)/2

def unique_symmetric_index2(i: int, j: int) -> int:
    min_ind = min(i, j)
    max_ind = max(i, j)
    diff = max_ind - min_ind
    return min_ind + diff * (min_ind + 1 + max_ind)/2

def unique_symmetric_index3(i: int, j: int) -> int:
    return max(i,j) * (max(i,j) + 1) / 2 + min(i,j)

def unique_symmetric_index4(i: int, j: int) -> int:
    diff = abs(i-j)
    return diff * (diff + 1) / 2


In [144]:
for i in range(5):
    for j in range(5):
        if j == 4:
            print(unique_symmetric_index4(i, j))
        else:
            print(unique_symmetric_index4(i, j), end=' ')

0.0 1.0 3.0 6.0 10.0
1.0 0.0 1.0 3.0 6.0
3.0 1.0 0.0 1.0 3.0
6.0 3.0 1.0 0.0 1.0
10.0 6.0 3.0 1.0 0.0
